# Notebook 17 — Reflected Manifold Decomposition

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Notebook 13 studied the pre-15 boundary lane `13` and its reflected partner `17`.

Notebook 17 now treats lane `17` as the post-15 reflected lane and decomposes rolling prime-lane vectors into covariance and spectral modes.

Constraint view:

> Lane 17 anchors the reflected side of the mod30 residue manifold, where rolling prime-lane trajectories can be decomposed into shared modes, reflected modes, and local deviations.


## Goals

1. Detect repo root and create standard output directories.
2. Load MRL foundations from Notebook 00 when available.
3. Generate rolling prime-filtered lane vectors.
4. Focus on lane `17` as reflected post-15 anchor.
5. Compute covariance, correlation, eigenmodes, and low-rank reconstructions.
6. Measure explained variance and lane loadings.
7. Export CSV, JSON, figures, and Markdown report.
8. Create a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "src" / "mod30_residue_lanes").exists() or (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Load Notebook 00 foundation output

In [ ]:
foundation_path = RESULTS_DIR / "notebook00_mrl_foundations.json"

if foundation_path.exists():
    foundation = json.loads(foundation_path.read_text())
    MODULUS = foundation["modulus"]
    ADMISSIBLE_RESIDUES = foundation["admissible_residues"]
    print("Loaded foundation:", foundation_path)
else:
    MODULUS = 30
    ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
    print("Notebook 00 foundation not found; using defaults.")

TARGET_LANE = 17
REFLECTED_LANE = 13
LANE_LABEL = f"{TARGET_LANE:02d}"
REFLECTED_LABEL = f"{REFLECTED_LANE:02d}"

print("MODULUS:", MODULUS)
print("ADMISSIBLE_RESIDUES:", ADMISSIBLE_RESIDUES)
print("TARGET_LANE:", LANE_LABEL)
print("REFLECTED_LANE:", REFLECTED_LABEL)


## Helpers

In [ ]:
def prime_sieve(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.zeros(n_max + 1, dtype=bool)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    limit = int(np.sqrt(n_max)) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n_max+1:p] = False

    return sieve


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


## Generate rolling prime lane vectors

In [ ]:
N_MAX = 120000
WINDOW_SIZE = 3000
STEP_SIZE = 300

values = np.arange(1, N_MAX + 1)
is_prime_array = prime_sieve(N_MAX)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_prime"] = is_prime_array[values]
df["is_admissible_prime"] = df["is_admissible"] & df["is_prime"]
df["cycle"] = (df["n"] - 1) // MODULUS

admissible_prime_df = df[df["is_admissible_prime"]].copy()
lane17_prime_df = df[df["is_admissible_prime"] & (df["residue"] == TARGET_LANE)].copy()

lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]
window_rows = []

for start in range(1, N_MAX - WINDOW_SIZE + 2, STEP_SIZE):
    stop = start + WINDOW_SIZE - 1
    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    prime_window = window[window["is_admissible_prime"]]
    counts = prime_window.groupby("residue").size().reindex(ADMISSIBLE_RESIDUES, fill_value=0)

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_center": (start + stop) / 2,
        "window_size": len(window),
        "prime_count": int(prime_window.shape[0]),
        "lane17_count": int(counts.loc[TARGET_LANE]),
        "lane13_count": int(counts.loc[REFLECTED_LANE]),
        "reflection_gap_17_13": int(counts.loc[TARGET_LANE] - counts.loc[REFLECTED_LANE]),
    }

    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)

    window_rows.append(row)

rolling_df = pd.DataFrame(window_rows)

rolling_csv = RESULTS_DIR / "notebook17_rolling_prime_lane_vectors.csv"
rolling_df.to_csv(rolling_csv, index=False)

print("Admissible prime values:", len(admissible_prime_df))
print("Lane 17 prime values:", len(lane17_prime_df))
print("Rolling windows:", len(rolling_df))
print(rolling_csv)

rolling_df.head()


## Centered matrix and covariance decomposition

The matrix rows are rolling windows; columns are the eight admissible lanes.

We center each lane trajectory before decomposition.


In [ ]:
X = rolling_df[lane_cols].to_numpy(dtype=float)
X_centered = X - X.mean(axis=0, keepdims=True)

cov = np.cov(X_centered, rowvar=False)
corr = np.corrcoef(X_centered, rowvar=False)

eigvals, eigvecs = np.linalg.eigh(cov)
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

explained = eigvals / eigvals.sum()
cumulative = np.cumsum(explained)

mode_rows = []
for k in range(len(eigvals)):
    row = {
        "mode": k + 1,
        "eigenvalue": float(eigvals[k]),
        "explained_variance": float(explained[k]),
        "cumulative_explained_variance": float(cumulative[k]),
    }
    for i, residue in enumerate(ADMISSIBLE_RESIDUES):
        row[f"loading_{residue:02d}"] = float(eigvecs[i, k])
    mode_rows.append(row)

modes_df = pd.DataFrame(mode_rows)

cov_df = pd.DataFrame(cov, index=[f"{r:02d}" for r in ADMISSIBLE_RESIDUES], columns=[f"{r:02d}" for r in ADMISSIBLE_RESIDUES])
corr_df = pd.DataFrame(corr, index=[f"{r:02d}" for r in ADMISSIBLE_RESIDUES], columns=[f"{r:02d}" for r in ADMISSIBLE_RESIDUES])

modes_csv = RESULTS_DIR / "notebook17_spectral_modes.csv"
cov_csv = RESULTS_DIR / "notebook17_lane_covariance_matrix.csv"
corr_csv = RESULTS_DIR / "notebook17_lane_correlation_matrix.csv"

modes_df.to_csv(modes_csv, index=False)
cov_df.to_csv(cov_csv)
corr_df.to_csv(corr_csv)

print(modes_csv)
print(cov_csv)
print(corr_csv)

modes_df.head()


## Low-rank reconstruction

We compare reconstruction error using the first `k` eigenmodes.


In [ ]:
reconstruction_rows = []

for k in range(1, len(ADMISSIBLE_RESIDUES) + 1):
    V_k = eigvecs[:, :k]
    scores_k = X_centered @ V_k
    X_hat = scores_k @ V_k.T + X.mean(axis=0, keepdims=True)

    error = np.linalg.norm(X - X_hat)
    relative_error = error / np.linalg.norm(X)

    reconstruction_rows.append({
        "modes_used": k,
        "frobenius_error": float(error),
        "relative_error": float(relative_error),
        "explained_variance": float(cumulative[k - 1]),
    })

reconstruction_df = pd.DataFrame(reconstruction_rows)

reconstruction_csv = RESULTS_DIR / "notebook17_low_rank_reconstruction.csv"
reconstruction_df.to_csv(reconstruction_csv, index=False)

print(reconstruction_csv)
reconstruction_df


## Mode scores over rolling windows

In [ ]:
scores = X_centered @ eigvecs

score_rows = []
for i, row in rolling_df.iterrows():
    out = {
        "window_start": int(row["window_start"]),
        "window_center": float(row["window_center"]),
    }
    for k in range(min(4, scores.shape[1])):
        out[f"mode_{k+1}_score"] = float(scores[i, k])
    score_rows.append(out)

scores_df = pd.DataFrame(score_rows)

scores_csv = RESULTS_DIR / "notebook17_mode_scores.csv"
scores_df.to_csv(scores_csv, index=False)

print(scores_csv)
scores_df.head()


## Lane 17 loading summary

In [ ]:
lane_index = ADMISSIBLE_RESIDUES.index(TARGET_LANE)
reflected_index = ADMISSIBLE_RESIDUES.index(REFLECTED_LANE)

loading_summary = pd.DataFrame([
    {
        "mode": k + 1,
        "lane17_loading": float(eigvecs[lane_index, k]),
        "lane13_loading": float(eigvecs[reflected_index, k]),
        "reflection_loading_gap_17_13": float(eigvecs[lane_index, k] - eigvecs[reflected_index, k]),
        "abs_lane17_loading": float(abs(eigvecs[lane_index, k])),
        "explained_variance": float(explained[k]),
    }
    for k in range(len(ADMISSIBLE_RESIDUES))
])

loading_csv = RESULTS_DIR / "notebook17_lane17_loading_summary.csv"
loading_summary.to_csv(loading_csv, index=False)

print(loading_csv)
loading_summary


## Summary exports

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "17_lane_residue_17",
    "title": "Reflected Manifold Decomposition",
    "modulus": MODULUS,
    "target_lane": TARGET_LANE,
    "target_lane_label": LANE_LABEL,
    "reflected_lane": REFLECTED_LANE,
    "reflected_lane_label": REFLECTED_LABEL,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "rolling_windows": int(len(rolling_df)),
    "admissible_prime_values": int(len(admissible_prime_df)),
    "lane17_prime_values": int(len(lane17_prime_df)),
    "mode1_explained_variance": float(explained[0]),
    "mode2_explained_variance": float(explained[1]),
    "mode3_explained_variance": float(explained[2]),
    "first_three_modes_cumulative_explained_variance": float(cumulative[2]),
    "mode1_lane17_loading": float(eigvecs[lane_index, 0]),
    "mode1_lane13_loading": float(eigvecs[reflected_index, 0]),
    "mean_reflection_gap_17_13": float(rolling_df["reflection_gap_17_13"].mean()),
    "std_reflection_gap_17_13": float(rolling_df["reflection_gap_17_13"].std(ddof=0)),
    "constraint_view": "Lane 17 anchors the reflected side of the mod30 residue manifold, where rolling prime-lane trajectories can be decomposed into shared modes, reflected modes, and local deviations.",
}

json_path = RESULTS_DIR / "notebook17_reflected_manifold_decomposition_summary.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)
    print(path)

# 1. Rolling heatmap
plt.figure(figsize=(11, 6))
plt.imshow(X, aspect="auto")
plt.xticks(range(len(lane_cols)), [c.replace("lane_", "") for c in lane_cols])
plt.yticks(
    range(0, len(rolling_df), max(1, len(rolling_df)//10)),
    rolling_df["window_start"].iloc[::max(1, len(rolling_df)//10)]
)
plt.xlabel("Residue lane")
plt.ylabel("Rolling window start")
plt.colorbar(label="Prime count")
plt.title("Notebook 17 — Rolling Prime Lane Matrix")
savefig("notebook17_rolling_prime_lane_matrix.png")

# 2. Correlation matrix
plt.figure(figsize=(7, 6))
plt.imshow(corr, vmin=-1, vmax=1)
plt.xticks(range(len(ADMISSIBLE_RESIDUES)), [f"{r:02d}" for r in ADMISSIBLE_RESIDUES])
plt.yticks(range(len(ADMISSIBLE_RESIDUES)), [f"{r:02d}" for r in ADMISSIBLE_RESIDUES])
plt.colorbar(label="Correlation")
plt.title("Notebook 17 — Lane Correlation Matrix")
savefig("notebook17_lane_correlation_matrix.png")

# 3. Explained variance
plt.figure(figsize=(8, 4))
plt.bar(range(1, len(explained) + 1), explained)
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
plt.xlabel("Mode")
plt.ylabel("Explained variance")
plt.title("Notebook 17 — Spectral Explained Variance")
savefig("notebook17_explained_variance.png")

# 4. First three eigenmodes
for mode_idx in range(3):
    plt.figure(figsize=(8, 4))
    plt.bar([f"{r:02d}" for r in ADMISSIBLE_RESIDUES], eigvecs[:, mode_idx])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Residue lane")
    plt.ylabel("Loading")
    plt.title(f"Notebook 17 — Eigenmode {mode_idx + 1} Lane Loadings")
    savefig(f"notebook17_eigenmode_{mode_idx + 1}_loadings.png")

# 5. Mode scores
plt.figure(figsize=(12, 5))
for k in range(3):
    plt.plot(scores_df["window_center"], scores_df[f"mode_{k+1}_score"], label=f"mode {k+1}")
plt.xlabel("Window center")
plt.ylabel("Mode score")
plt.title("Notebook 17 — First Three Mode Scores")
plt.legend()
savefig("notebook17_mode_scores.png")

# 6. Reconstruction error
plt.figure(figsize=(8, 4))
plt.plot(reconstruction_df["modes_used"], reconstruction_df["relative_error"], marker="o")
plt.xlabel("Modes used")
plt.ylabel("Relative reconstruction error")
plt.title("Notebook 17 — Low-Rank Reconstruction Error")
savefig("notebook17_low_rank_reconstruction_error.png")

# 7. Lane 17 vs 13 rolling counts
plt.figure(figsize=(12, 5))
plt.plot(rolling_df["window_center"], rolling_df["lane17_count"], label="17")
plt.plot(rolling_df["window_center"], rolling_df["lane13_count"], label="13")
plt.xlabel("Window center")
plt.ylabel("Prime count")
plt.title("Notebook 17 — Reflected Pair Counts: 17 vs 13")
plt.legend()
savefig("notebook17_lane17_vs_lane13.png")

# 8. Reflection gap 17 - 13
plt.figure(figsize=(12, 5))
plt.plot(rolling_df["window_center"], rolling_df["reflection_gap_17_13"])
plt.axhline(0, linewidth=1)
plt.xlabel("Window center")
plt.ylabel("Lane 17 count minus lane 13 count")
plt.title("Notebook 17 — Reflection Gap 17 − 13")
savefig("notebook17_reflection_gap_17_13.png")


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
report_path = REPORTS_DIR / "report_17_reflected_manifold_decomposition.md"

output_links = "\n".join([
    '- Rolling prime lane vectors CSV: <a href="../results/notebook17_rolling_prime_lane_vectors.csv">`results/notebook17_rolling_prime_lane_vectors.csv`</a>',
    '- Spectral modes CSV: <a href="../results/notebook17_spectral_modes.csv">`results/notebook17_spectral_modes.csv`</a>',
    '- Lane covariance matrix CSV: <a href="../results/notebook17_lane_covariance_matrix.csv">`results/notebook17_lane_covariance_matrix.csv`</a>',
    '- Lane correlation matrix CSV: <a href="../results/notebook17_lane_correlation_matrix.csv">`results/notebook17_lane_correlation_matrix.csv`</a>',
    '- Low-rank reconstruction CSV: <a href="../results/notebook17_low_rank_reconstruction.csv">`results/notebook17_low_rank_reconstruction.csv`</a>',
    '- Mode scores CSV: <a href="../results/notebook17_mode_scores.csv">`results/notebook17_mode_scores.csv`</a>',
    '- Lane 17 loading summary CSV: <a href="../results/notebook17_lane17_loading_summary.csv">`results/notebook17_lane17_loading_summary.csv`</a>',
    '- Summary JSON: <a href="../results/notebook17_reflected_manifold_decomposition_summary.json">`results/notebook17_reflected_manifold_decomposition_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 17 — Reflected Manifold Decomposition

Notebook 17 treats lane `17` as the post-15 reflected lane and decomposes rolling prime-lane vectors into covariance and spectral modes.

Constraint view:

> Lane 17 anchors the reflected side of the mod30 residue manifold, where rolling prime-lane trajectories can be decomposed into shared modes, reflected modes, and local deviations.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Target lane | {LANE_LABEL} |
| Reflected lane | {REFLECTED_LABEL} |
| Interval max | {N_MAX} |
| Window size | {WINDOW_SIZE} |
| Step size | {STEP_SIZE} |
| Rolling windows | {len(rolling_df)} |
| Admissible prime values | {len(admissible_prime_df)} |
| Lane 17 prime values | {len(lane17_prime_df)} |
| Mode 1 explained variance | {explained[0]:.6f} |
| Mode 2 explained variance | {explained[1]:.6f} |
| Mode 3 explained variance | {explained[2]:.6f} |
| First three modes cumulative explained variance | {cumulative[2]:.6f} |
| Mean reflection gap 17 − 13 | {rolling_df["reflection_gap_17_13"].mean():.6f} |

## Spectral modes

{modes_df.to_markdown(index=False)}

## Low-rank reconstruction

{reconstruction_df.to_markdown(index=False)}

## Lane 17 loading summary

{loading_summary.to_markdown(index=False)}

## Interpretation

- Notebook 13 measured boundary pressure around the `13 | 17` split.
- Notebook 17 treats lane `17` as a reflected post-15 anchor.
- Covariance and correlation matrices identify coupled lane neighborhoods.
- Eigenmodes expose shared and opposing components in rolling prime-lane trajectories.
- Low-rank reconstruction tests whether a few modes recover most observable structure.

## Next step

Notebook 19 can construct transition operators between rolling lane states and study lane-to-lane transition dynamics.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])


## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook17_reflected_manifold_decomposition_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook17_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook17_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_17_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook17_reflected_manifold_decomposition_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook17_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_17_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
